# Detection Threshold

Compare face probs on real faces vs random objects. `detection_prob_threshold = 0.87`.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import numpy as np
import matplotlib.pyplot as plt
from facenet_models import FacenetModel
from core.normalize import load_image, resize_image

%matplotlib inline

In [ ]:
root = Path.cwd().parent / "data" / "test_images"
face_dirs = [root / "Team's Faces", root / "Online Faces"]
object_dir = root / "Random Objects (Noise)"
exts = {".jpg", ".jpeg", ".png"}


def get_probs(folder):
    probs = []
    for path in sorted(folder.iterdir()):
        if path.suffix.lower() not in exts:
            continue
        img = resize_image(load_image(path))
        boxes, p, _ = model.detect(img)
        if boxes is None:
            continue
        probs.extend(float(x) for x in p if x is not None)
    return np.array(probs)


model = FacenetModel()

face_probs = np.concatenate([get_probs(d) for d in face_dirs])
object_probs = get_probs(object_dir)

print("face detections:", len(face_probs), "min:", round(face_probs.min(), 3))
print("object detections:", len(object_probs), "max below 0.9:", round(object_probs[object_probs < 0.9].max(), 3))

In [ ]:
detection_prob_threshold = 0.87
bins = np.linspace(0.65, 1.0, 15)

fig, ax = plt.subplots(figsize=(9, 5))
ax.hist(face_probs, bins=bins, alpha=0.65, label="faces", color="#2a6fbb")
ax.hist(object_probs, bins=bins, alpha=0.65, label="random objects", color="#c44e52")
ax.axvline(detection_prob_threshold, color="black", ls="--", lw=2, label=f"threshold = {detection_prob_threshold}")
ax.set_xlabel("detection probability")
ax.set_ylabel("count")
ax.set_title("Face detection probabilities")
ax.legend()
fig.tight_layout()

out = Path.cwd() / "detection_threshold_histogram.png"
fig.savefig(out, dpi=300, bbox_inches="tight")
plt.show()
print("saved:", out)

In [ ]:
detection_prob_threshold